In [ ]:
using Plots

using PyCall
@pyimport umap as umap;

include("../src/fxio.jl")
include("../src/kabsch_umeyama.jl")

In [ ]:
f = open("../data/dssp.csv") 
dssp = Dict()
for line in readlines(f) 
    key_seq = split(line, ",")
    dssp[key_seq[1]] = key_seq[2]
end

In [ ]:
function convertdssp(dssp4mer)

    for i in 1:length(dssp4mer)
        if contains(dssp4mer[i], "HHH")
            dssp4mer[i] = "red"
        elseif  contains(dssp4mer[i], "EE")
            dssp4mer[i] = "blue"
        elseif  contains(dssp4mer[i], "TT")
            dssp4mer[i] = "green"
        else 
            dssp4mer[i] = "gray"
        end
    end
    
    return dssp4mer
end

function pdb2vtors(pdbpath)
    seqxyz = pdb2seqxyz(pdbpath)
    kmers = coords2kmers(seqxyz, 4, "ca")

    vtors = zeros(Float64, length(kmers), 3)
    
    for (i, kmer) in enumerate(kmers)
        vtors[i,1:3] = vtor(kmer[:,2:4])
    end
    
    return vtors
end


In [ ]:
scoppath = "../../scop40NR/" 
virtual_torsions = reshape(Float64[], 0, 3)
sscolors = []
for pdb in readdir(scoppath)

    try
        pdbpath = joinpath(scoppath, pdb)
        id = basename(pdbpath)[1:end-4]
        dsspcolors = convertdssp(seq2kmers(dssp[id], 4))
        vtors = pdb2vtors(pdbpath)   
        if (size(vtors, 1) == length(dsspcolors)) 
            if !any(isnan.(vtors))
            virtual_torsions = vcat(virtual_torsions, vtors)
            sscolors = vcat(sscolors, dsspcolors)
            end
        end
    catch

        println(pdb)#, " ", size(vtors, 1), " ", length(dsspcolors))
    
    end
end

println(size(virtual_torsions, 1))
println(length(sscolors))

In [ ]:
range = 1:10000
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.01, n_epochs=200)[:fit_transform](permutedims(virtual_torsions[range,:]')) 
print(size(umap_model))

In [ ]:
scatter(umap_model[:,1], umap_model[:,2], color=sscolors[range], title="Virtual torsions UMAP", markersize=3, alpha=0.5)